## Imports

In [1]:
from functools import partial
from pathlib import Path

import numpy as np

from boamotion import Boa
from compare_macro_utils import compare_results_files, load_macro_log, write_or_print

## Define input and output

General paths:

In [2]:
in_dir = r"C:\Users\roman\OneDrive - Universitaet Bern\MuscleMotion\synthetic_blobs_dataset"
# A folder of this name is created next to the input folder
out_folder = "synthetic_blobs_dataset-results"
# Additionally write the comparison to a file in there, not only print it
write_to_file = False

In [3]:
in_path = Path(in_dir)
out_path = in_path.parent / out_folder  # create output besides the input folder
file_dir = out_path if write_to_file else None
# Bind the destination once, so every report below goes the same way
report = partial(write_or_print, file_dir=file_dir)

The subfolder inside out_folder for the BoaMotion results:

In [4]:
analysis_name = "MM_Boa_diff_4"  # NOTE: legacy=True adds "-Contr-Results" to the outputs folder

The folder where the macro results are stored:

In [5]:
macro_dir = out_path / "26-08-31_Fiji_default" / "frame_0001-Contr-Results"

## Analyse with BoaMotion

Set the parameters for BoaMotion. The parameters that are not set here will be set to their default values.

In [6]:
non_defaults = {"framerate": 25, "peak_window": 16, "legacy": True}

In [7]:
boa_run = Boa(in_dir, name=analysis_name)
boa_run.set_params(**non_defaults)

report("=== BOA PARAMETERS ===\n")
for k, v in boa_run.params.to_dict().items():
    report(f"{k:20} {v}")

=== BOA PARAMETERS ===

framerate            25
speed_window         2
reference_frame      None
ref_search_start     1
ref_search_stop      300
n_low_values         20
n_unity_values       10
noise_reduction      True
mask_start_frame     1
mask_end_frame       None
transient_analysis   True
peak_window          16
peak_threshold       30.0
percentages          [10, 50, 90]
flank_level_index    0
baseline_threshold   2.0
baseline_n_points    5
high_freq_baseline   True
legacy               True


In [8]:
res = boa_run.run()

In [9]:
# NOTE: legacy=True adds "-Contr-Results" to the folder name, so boa_dir is
# the path actually written rather than out_path itself
boa_dir = res.save(out_path)

## Compare the outputs with the FIJI plugin results (ran separately)

### 1) Log_file

In [10]:
macro_log = load_macro_log(macro_dir / "Log_file.txt")
s = macro_log["settings"]  # The parameters that were used in the macro run
p = res.params  # The logged - i.e., actually used - parameters from the BoaMotion run

In [11]:
report("=== LOG FILE ===\n")

lines = 5
report("Macro log file:")
report("---")
for k, v in macro_log.items():
    if lines <= 0:
        break
    report(k, " " * (30 - len(str(k))) + str(v))
    lines -= 1
report("...", "\n")
report("Macro system parameters:", s)
report("\nBoa execution log:")
report("---")
report("Boa results object:", res)
report("Boa processed parameters:", p)

=== LOG FILE ===

Macro log file:
---
recording                      frame_0001
version                        1.0
reference_frame                6
peaks                          (14, 39, 64, 89)
percentages                    (10, 50, 90)
... 

Macro system parameters: {'recordedFramerate': 25, 'speedWindow': 2, 'referenceFrameSlice': 1, 'maxProject': 1, 'MPstartRange': 1, 'MPendRange': -1, 'hideIntermediateResults': 1, 'checkClip': 1, 'checkSpeedlinearity': 1, 'autodetectReferenceFrame': 1, 'lowValueN': 20, 'unitySelectionN': 10, 'autoDetectStart': 1, 'autoDetectStop': 300, 'manualReferenceFrame': 0, 'automaticTransientDetection': 1, 'PeakDetectionWindow': 16, 'peakThreshold': 30, 'drawPeaks': 1, 'baselineThreshold': 2, 'baselineNumberOfPoints': 5, 'highFreqBaselineDetection': 1, 'guassianBlur10': 'No', 'tiffImSequence': 'Yes', 'batchDirLoad': 0}

Boa execution log:
---
Boa results object: Result('MM_Boa_diff_4': 99 points, reference frame 6, 4 beats, masked)
Boa processed parameters

In [12]:
def no_flank(column):
    """Beats where that measurement is missing, as 1-based numbers."""
    return tuple(i + 1 for i, missing in enumerate(res.beats[column].isna()) if missing)


# The macro prints "lowUp false" only where the rising flank was found, so for a beat that
# lost both it says nothing about the falling one. Compare where the log can speak.
unstated = set(macro_log["unmeasured_flanks"]["rising"])
falling = tuple(beat for beat in no_flank("relaxation_time_ms") if beat not in unstated)

checks = [
    ("reference frame", macro_log["reference_frame"], res.reference_frame),
    ("beats", len(macro_log["peaks"]), res.n_beats),
    ("peak positions", macro_log["peaks"], tuple(res.beats["peak_position"])),
    ("no rising flank", macro_log["unmeasured_flanks"]["rising"], no_flank("time_to_peak_ms")),
    ("no falling flank", macro_log["unmeasured_flanks"]["falling"], falling),
    ("percentages", macro_log["percentages"], p.percentages),
    ("framerate", s["recordedFramerate"], p.framerate),
    ("speed window", s["speedWindow"], p.speed_window),
    ("peak window", s["PeakDetectionWindow"], p.peak_window),
    ("peak threshold", s["peakThreshold"], p.peak_threshold),
    ("baseline threshold", s["baselineThreshold"], p.baseline_threshold),
    ("baseline points", s["baselineNumberOfPoints"], p.baseline_n_points),
    ("high-freq baseline", bool(s["highFreqBaselineDetection"]), p.high_freq_baseline),
    ("noise reduction", bool(s["maxProject"]), p.noise_reduction),
    ("gaussian blur", s["guassianBlur10"], "No"),
]

In [13]:
report("\nMacro log vs. BOA log:")
report("---")
for label, macro, ours in checks:
    if macro == ours:
        report(f"ok    {label}")
    else:
        report(f"DIFF  {label}: macro {macro!r}  vs  boa {ours!r}")


Macro log vs. BOA log:
---
ok    reference frame
ok    beats
ok    peak positions
ok    no rising flank
ok    no falling flank
ok    percentages
ok    framerate
ok    speed window
ok    peak window
ok    peak threshold
ok    baseline threshold
ok    baseline points
ok    high-freq baseline
ok    noise reduction
ok    gaussian blur


### 2) Three results files

- Overview-results.txt
- contraction.txt
- speed-of-contraction.txt

In [14]:
results_files = ("Overview-results.txt", "contraction.txt", "speed-of-contraction.txt")
rel_thresh = 1e-6

In [15]:
report("=== RESULTS FILES ===\n")

for name in results_files:
    report(name, "\n")

    macro = np.loadtxt(macro_dir / name, ndmin=2)
    ours = np.loadtxt(boa_dir / name, ndmin=2)
    worst, relative, verdict, column = compare_results_files(macro, ours, rel_thresh)

    if verdict is None:
        report(f"DIFF SHAPE {name}: macro {macro.shape} vs boa {ours.shape}")
        continue

    report(
        f"{verdict}: {macro.shape[0]} rows, worst column {column}, "
        f"largest difference {worst:g} ({relative:.1e} relative)"
    )

    report()
    report("Macro file:")
    report(macro[:2, :3])
    report("Boa file:")
    report(ours[:2, :3])
    report("\n", "=== === ===", "\n")

=== RESULTS FILES ===

Overview-results.txt 

OK: 4 rows, worst column 0, largest difference 0 (0.0e+00 relative)

Macro file:
[[480. 200. 280.]
 [480. 200. 280.]]
Boa file:
[[480. 200. 280.]
 [480. 200. 280.]]

 === === === 

contraction.txt 

OK: 99 rows, worst column 1, largest difference 0.016 (4.3e-08 relative)

Macro file:
[[    0.     14171.127 ]
 [   40.     14587.3447]]
Boa file:
[[    0.     14171.127 ]
 [   40.     14587.3447]]

 === === === 

speed-of-contraction.txt 

OK: 97 rows, worst column 1, largest difference 0.008 (4.1e-08 relative)

Macro file:
[[    0.     14144.083 ]
 [   40.     14155.5381]]
Boa file:
[[    0.     14144.083 ]
 [   40.     14155.5381]]

 === === === 

